# Module 10 - 실습 2 솔루션: Annotated Reducer 패턴

In [ ]:
import sys
sys.path.insert(0, '../..')

import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, END
from common.fake_llm import FakeLLM

print("Annotated Reducer 솔루션 준비 완료")

## 실습 1 솔루션: _replace reducer

In [ ]:
def _replace(existing, new):
    """기존 값을 새 값으로 교체하는 reducer."""
    return new

In [ ]:
assert _replace("old", "new") == "new"
assert _replace(None, "value") == "value"
assert _replace([1, 2], [3, 4]) == [3, 4]
print("✅ 실습 1 완료! _replace reducer가 올바릅니다.")

## 실습 2 솔루션: Annotated Reducer State

In [ ]:
class ChatState(TypedDict):
    query: Annotated[str, _replace]
    messages: Annotated[list[str], operator.add]  # 리스트 누적
    current_step: Annotated[str, _replace]

## 실습 3 솔루션: 그래프 구현

In [ ]:
def greet_node(state: ChatState) -> dict:
    """인사 메시지를 추가합니다."""
    return {
        "messages": ["안녕하세요!"],  # operator.add로 기존 messages에 추가됨
        "current_step": "greet",
    }

def ask_node(state: ChatState) -> dict:
    """질문 메시지를 추가합니다."""
    return {
        "messages": ["무엇을 도와드릴까요?"],
        "current_step": "ask",
    }

def respond_node(state: ChatState) -> dict:
    """응답 메시지를 추가합니다."""
    response = f"'{state['query']}'에 대해 답변드립니다."
    return {
        "messages": [response],
        "current_step": "respond",
    }

# 그래프 구성
graph = StateGraph(ChatState)
graph.add_node("greet", greet_node)
graph.add_node("ask", ask_node)
graph.add_node("respond", respond_node)

graph.set_entry_point("greet")
graph.add_edge("greet", "ask")
graph.add_edge("ask", "respond")
graph.add_edge("respond", END)

app = graph.compile()
print("그래프 구성 완료")

In [ ]:
# 검증
initial_state = {
    "query": "Python에 대해 알려주세요",
    "messages": [],
    "current_step": "start",
}

result = app.invoke(initial_state)

assert "messages" in result
assert len(result["messages"]) >= 3
assert result["current_step"] == "respond"

print("메시지 목록:")
for i, msg in enumerate(result["messages"]):
    print(f"  {i+1}. {msg}")
print("✅ 실습 3 완료! Annotated Reducer 패턴이 올바르게 작동합니다.")

## 실습 4 솔루션: operator.add vs _replace 비교

In [ ]:
existing_list = ["메시지1", "메시지2"]
new_list = ["메시지3"]
add_result = operator.add(existing_list, new_list)
replace_result = _replace(existing_list, new_list)

print(f"operator.add: {existing_list} + {new_list} = {add_result}")
print(f"_replace:     {existing_list} -> {new_list} = {replace_result}")

assert add_result == ["메시지1", "메시지2", "메시지3"]
assert replace_result == ["메시지3"]
print("✅ 실습 4 완료! operator.add vs _replace 차이를 이해했습니다.")